# Model 1 — Kilometers_Driven regression: experiments (Sprint 3)

**Goal.** Choose the data and the algorithm for the Stage-1 regression that estimates a vehicle's odometer reading
from what the live system can actually know (data.gov.il registry, vehicle specs, PostgreSQL).

**Scope of this notebook: experiments only.** No final training, no `.joblib` export, no backend integration —
those wait for approval. The old `.joblib` files from the first attempt are **not** used.

**Questions answered here**
1. Are datasets 7 (UK), 8 (India), 3 (Cardekho) clean and consistent? (rows, nulls, duplicates, units, snapshot year)
2. Which features will really exist at runtime, and how much does each add?
3. Encoding (OHE vs Target vs Frequency) and algorithm (Ridge vs Random Forest vs XGBoost) comparison.
4. **7+8+3 vs 7+8+3+6 (Craigslist)** on the *same* held-out test set, without leakage, in RMSE / MAE / R²,
   including a metric re-weighted to the Israeli registry and a measure of Israeli compatibility.

**Leakage controls.** (a) exact duplicates removed *inside each source before any split*; (b) one fixed test set drawn
from the 7+8+3 pool only — Dataset 6 can only ever enter training; (c) target encoders live inside the sklearn
`Pipeline`, so they are fit on training folds only; (d) the test set is not used to choose features/encodings
(those are chosen by validation/CV on the training part).

In [1]:
import os, sys, hashlib, warnings, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.options.display.width = 170
pd.options.display.max_columns = 40
pd.options.display.float_format = "{:,.3f}".format
sys.path.insert(0, str(Path.cwd().parent / "src"))
from config import RAW_DIR, INTERIM_DIR, MILES_TO_KM
import mileage_data as md_
import mileage_experiments as mx
from israel_registry import normalise_registry
from sklearn.model_selection import train_test_split, KFold, cross_validate
try:
    display
except NameError:
    display = print
SMOKE = os.environ.get("SMARTCAR_SMOKE") == "1"   # quick dry-run on subsamples (used only while developing)
import sklearn, xgboost
print("python", sys.version.split()[0], "| sklearn", sklearn.__version__, "| xgboost", xgboost.__version__, "| pandas", pd.__version__, "| SMOKE =", SMOKE)

python 3.13.13 | sklearn 1.9.0 | xgboost 3.4.1 | pandas 3.0.5 | SMOKE = False


## 1. Data provenance and audit

In [2]:
def sha256(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while b := f.read(chunk):
            h.update(b)
    return h.hexdigest()[:16]

files = sorted(RAW_DIR.glob("d7_uk_cars/*.csv")) + sorted(RAW_DIR.glob("d8_used_cars_in/*.csv")) + sorted(RAW_DIR.glob("d3_cardekho/*.csv")) + sorted(RAW_DIR.glob("d6_craigslist/*.csv"))
files = [p for p in files if "unclean" not in p.name and p.name not in ("cclass.csv", "focus.csv")]
prov = pd.DataFrame([{"file": str(p.relative_to(RAW_DIR)), "MB": p.stat().st_size / 1e6, "sha256[:16]": sha256(p)} for p in files])
display(prov)

,file,MB,sha256[:16]
0,d7_uk_cars\audi.csv,0.534,f41fcda6795e500d
1,d7_uk_cars\bmw.csv,0.593,7efa367e8c108a7d
2,d7_uk_cars\ford.csv,0.933,48465d469ce740f8
3,d7_uk_cars\hyundi.csv,0.248,b481695ec0c926da
4,d7_uk_cars\merc.csv,0.741,8091d8c75199db37
5,d7_uk_cars\skoda.csv,0.335,cf7595b8f9bb1274
6,d7_uk_cars\toyota.csv,0.348,078f3d68d4583c5a
7,d7_uk_cars\vauxhall.csv,0.715,1a09a3440ab93e25
8,d7_uk_cars\vw.csv,0.788,f560dc91f9ab752b
9,d8_used_cars_in\test-data.csv,0.129,fde4a6e354d70522


**Notes on the files.** `cclass.csv` / `focus.csv` (and the `unclean *` files) of Dataset 7 are model-specific extracts of
`merc.csv` / `ford.csv`, so only the 9 brand files are used (avoids double counting).
The Kaggle slug of **Dataset 6 now serves a newer scrape (posts from Sep–Nov 2026, ~295k rows)** than the one the
research PDF described (~426k rows, 2021) — hence the hash table above, for reproducibility.

In [3]:
clean, logs = {}, {}
for name in md_.LOADERS:
    df, log = md_.load_clean(name)
    if SMOKE and len(df) > 20000:
        df = df.sample(20000, random_state=0).reset_index(drop=True)
    clean[name], logs[name] = df, log
audit = pd.DataFrame(logs).fillna(0).astype(int)
display(audit)
print()
for k, df in clean.items():
    print(f"{k:14s} rows={len(df):>7,}  nulls(make/age/fuel/km)={int(df[['make','age','fuel','km']].isna().sum().sum())}  "
          f"transmission NaN={df.transmission.isna().mean():.1%}  makes={df.make.nunique()}  fuels={df.fuel.value_counts().to_dict()}")

,d7_uk,d8_in,d3_cardekho,d6_craigslist
raw_rows,99187,7253,15411,295451
missing make/age/fuel/km (or unmapped fuel),247,0,0,21282
"age outside [0,25]",3,0,1,20666
"km outside (0,500,000]",1,4,12,1082
"implausible rate > 100,000 km/yr",2,3,3,354
"age>=3 with <1,000 km (typo/placeholder)",43,0,0,1529
exact duplicate listing (all attributes + km),5311,50,2313,97147
clean_rows,93580,7196,13082,153391



d7_uk          rows= 93,580  nulls(make/age/fuel/km)=0  transmission NaN=0.0%  makes=9  fuels={'petrol': 51646, 'diesel': 38930, 'hybrid': 2998, 'electric': 6}
d8_in          rows=  7,196  nulls(make/age/fuel/km)=0  transmission NaN=0.0%  makes=32  fuels={'diesel': 3816, 'petrol': 3304, 'lpg_cng': 74, 'electric': 2}
d3_cardekho    rows= 13,082  nulls(make/age/fuel/km)=0  transmission NaN=0.0%  makes=31  fuels={'petrol': 6474, 'diesel': 6276, 'lpg_cng': 328, 'electric': 4}
d6_craigslist  rows=153,391  nulls(make/age/fuel/km)=0  transmission NaN=1.9%  makes=40  fuels={'petrol': 138224, 'diesel': 8418, 'hybrid': 4319, 'electric': 2430}


### 1.1 Units and snapshot year (two real bugs found in the first attempt)
* **UK `mileage` and Craigslist `odometer` are in miles**; India (D8/D3) is km. The first attempt mixed them. All are now km (×1.609344).
* Odometer is a **snapshot at listing time**. Age must be `snapshot_year − manufacture_year` (UK 2020, India-D8 2019, Cardekho ships `vehicle_age`,
  Craigslist uses each post's own `posting_date`) — not `2026 − year`, which would turn a 2020 listing of a 2020 car into a "6-year-old car with almost no km".
* **Duplicates**: the cleaning log shows how many rows were re-posted listings (Craigslist: ~1/3 of rows).

In [4]:
allc = pd.concat(clean.values(), ignore_index=True)
allc["annual_km"] = allc.km / allc.age.clip(lower=1)
sub = allc[allc.age.between(1, 8)]
summary = sub.groupby("source").agg(n=("km", "size"), median_annual_km=("annual_km", "median"), median_age=("age", "median"))
summary["if_miles_were_(wrongly)_taken_as_km"] = np.where(summary.index.isin(["d7_uk", "d6_craigslist"]), summary.median_annual_km / MILES_TO_KM, summary.median_annual_km)
display(summary)

med = allc[allc.age.between(0, 15)].pivot_table(index="age", columns="source", values="km", aggfunc="median")
fig, ax = plt.subplots(figsize=(8, 4.5))
for c in med.columns:
    ax.plot(med.index, med[c], marker="o", ms=3, label=c)
ages = np.arange(0, 16)
ax.plot(ages, 16100 * ages, "k--", label="Israel CBS avg 16,100 km/yr × age (rough anchor)")
ax.set_xlabel("vehicle age (years)"); ax.set_ylabel("median odometer (km)"); ax.set_title("Median odometer by age, per source (km)")
ax.legend(fontsize=8); ax.grid(alpha=.3); plt.show()

,n,median_annual_km,median_age,if_miles_were_(wrongly)_taken_as_km
source,,,,
d3_cardekho,10373,"9,100.000",5.000,"9,100.000"
d6_craigslist,52818,"23,192.127",6.000,"14,410.920"
d7_uk,88699,"11,426.342",3.000,"7,100.000"
d8_in,5824,"11,000.000",5.000,"11,000.000"


**External anchor for Israel** (not ground truth, only a plausibility check): the Israeli Central Bureau of Statistics reports an average of
~16,100 km/year per private car in 2018 (≈20,600 km/yr for cars up to 3 years old; ≈14,600 km/yr for petrol/diesel in a 2024 survey).
Sources: [Jerusalem Post / CBS 2018](https://www.pressreader.com/israel/jerusalem-post/20190917/281672551649195),
[Ynet — EV owners drive 40% more](https://www.ynetnews.com/business/article/rjgb0rvma).
Implied annual km above: UK ≈ 11k, India ≈ 9–11k, **US Craigslist ≈ 23k** → Israel sits *between* the European/Indian and the US usage regimes.

### 1.2 Same *shape*, different *level*? (and a selection-bias warning)
Median odometer by age divided by the median at age 5. If the markets share the shape of the age curve and differ mainly in level, a market-level correction is conceptually justified.
**Warning:** all four datasets are **used-car listings** (mileage at the moment of sale), not a census of the fleet in circulation, so even a perfect fit would describe
"mileage at sale", not "mileage of every registered car". Leasing/company cars (13% of the Israeli fleet) are typically driven more and turned over earlier.

In [5]:
shape = (med.loc[1:12] / med.loc[5]).T
display(shape.round(2))
slopes = {s_: np.polyfit(g_.age, g_.km, 1)[0] for s_, g_ in allc[allc.age.between(1, 12)].groupby("source")}
display(pd.Series(slopes, name="linear slope, km per year of age (ages 1-12)").to_frame())

age,1.000,2.000,3.000,4.000,5.000,6.000,7.000,8.000,9.000,10.000,11.000,12.000
source,,,,,,,,,,,,
d3_cardekho,0.140,0.300,0.580,0.800,1.000,1.130,1.200,1.280,1.340,1.360,1.480,1.400
d6_craigslist,0.240,0.400,0.580,0.840,1.000,1.130,1.230,1.290,1.400,1.470,1.560,1.620
d7_uk,0.160,0.380,0.590,0.780,1.000,1.150,1.320,1.640,1.830,2.020,2.100,2.320
d8_in,0.400,0.530,0.720,0.910,1.000,1.080,1.180,1.180,1.270,1.270,1.420,1.370


,"linear slope, km per year of age (ages 1-12)"
d3_cardekho,"6,469.135"
d6_craigslist,"14,521.458"
d7_uk,"12,849.987"
d8_in,"5,522.218"


## 2. What the live system will see: a random sample of the Israeli registry

In [6]:
ilraw = pd.read_csv(INTERIM_DIR / "gov_registry_sample.csv")
il = normalise_registry(ilraw, reference_year=2026)
print(f"{len(il):,} passenger vehicles sampled at random from the registry (resource 053cea08…, same one the backend uses)")
share = il.make.value_counts(normalize=True)
print(f"unmapped makes (not in my Hebrew→English dictionary): {share.get('Unmapped', 0):.1%}")
display(pd.DataFrame({"share": share.head(18)}))
print("fuel   :", il.fuel.value_counts(normalize=True).round(3).to_dict())
print("owner. :", il.ownership.value_counts(normalize=True).round(3).to_dict())
print("age q  :", il.age.quantile([.05, .25, .5, .75, .95]).to_dict())

19,200 passenger vehicles sampled at random from the registry (resource 053cea08…, same one the backend uses)
unmapped makes (not in my Hebrew→English dictionary): 1.8%


,share
make,
Toyota,0.156
Kia,0.142
Hyundai,0.130
Mazda,0.052
Skoda,0.051
Suzuki,0.050
Nissan,0.047
Renault,0.032
Mitsubishi,0.027


fuel   : {'petrol': 0.83, 'electric': 0.068, 'diesel': 0.061, 'hybrid': 0.037, 'lpg_cng': 0.004}
owner. : {'private': 0.829, 'leasing': 0.082, 'company': 0.048, 'dealer': 0.028, 'rental': 0.014}
age q  : {0.05: 0.0, 0.25: 3.0, 0.5: 7.0, 0.75: 12.0, 0.95: 18.0}


In [7]:
MIN_SUPPORT = 200   # a category/age counts as "covered" only if the pool has >= 200 rows for it
def coverage(pool):
    mk, fu = pool.make.value_counts(), pool.fuel.value_counts()
    ag = pool.age.clip(upper=25).round().value_counts()
    okm, okf, oka = set(mk[mk >= MIN_SUPPORT].index), set(fu[fu >= MIN_SUPPORT].index), set(ag[ag >= MIN_SUPPORT].index)
    m, f, a = il.make.isin(okm), il.fuel.isin(okf), il.age.clip(0, 25).round().isin(oka)
    return {"rows": len(pool), "make": m.mean(), "fuel": f.mean(), "age": a.mean(), "make&fuel&age": (m & f & a).mean()}

pools = {
    "D7 UK": clean["d7_uk"], "D8 India": clean["d8_in"], "D3 Cardekho": clean["d3_cardekho"],
    "D7+D8+D3": pd.concat([clean["d7_uk"], clean["d8_in"], clean["d3_cardekho"]]),
    "D6 Craigslist": clean["d6_craigslist"],
    "D7+D8+D3+D6": pd.concat([clean["d7_uk"], clean["d8_in"], clean["d3_cardekho"], clean["d6_craigslist"]]),
}
cov = pd.DataFrame({k: coverage(v) for k, v in pools.items()}).T
display(cov)
print("Reading: share of the Israeli passenger fleet whose make / fuel / age (and all three) has >= 200 training rows in the pool.")

,rows,make,fuel,age,make&fuel&age
D7 UK,"93,580.000",0.401,0.929,0.733,0.278
D8 India,"7,196.000",0.461,0.891,0.645,0.277
D3 Cardekho,"13,082.000",0.487,0.895,0.718,0.336
D7+D8+D3,"113,858.000",0.502,0.932,0.838,0.400
D6 Craigslist,"153,391.000",0.686,0.996,1.000,0.683
D7+D8+D3+D6,"267,249.000",0.829,1.000,1.000,0.829


Reading: share of the Israeli passenger fleet whose make / fuel / age (and all three) has >= 200 training rows in the pool.


In [8]:
bins = np.arange(0, 27)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(il.age.clip(0, 26), bins=bins, density=True, alpha=.45, label="Israel registry (age today)", color="k")
for k in ["D7+D8+D3", "D6 Craigslist"]:
    ax.hist(pools[k].age.clip(0, 26), bins=bins, density=True, histtype="step", lw=2, label=k)
ax.set_xlabel("age (years)"); ax.set_title("Age support: Israeli fleet vs training pools"); ax.legend(); plt.show()

mk_share = pd.DataFrame({
    "Israel": il.make.value_counts(normalize=True),
    "D7+D8+D3": pools["D7+D8+D3"].make.value_counts(normalize=True),
    "D6": pools["D6 Craigslist"].make.value_counts(normalize=True),
}).fillna(0).sort_values("Israel", ascending=False).head(16)
display(mk_share)

,Israel,D7+D8+D3,D6
make,,,
Toyota,0.156,0.067,0.099
Kia,0.142,0.000,0.026
Hyundai,0.130,0.074,0.028
Mazda,0.052,0.000,0.014
Skoda,0.051,0.057,0.000
Suzuki,0.050,0.049,0.000
Nissan,0.047,0.001,0.058
Renault,0.032,0.005,0.000
Mitsubishi,0.027,0.000,0.005


### 2.1 `Owner_Type` (Dataset 8) is **not** the registry's `baalut`
The research PDF assumes Dataset 8's `Owner_Type` maps to the registry field `baalut`. They are different concepts:
`Owner_Type` = *how many previous owners* (First/Second/Third…), whereas `baalut` = *ownership category*
(private / leasing / company / dealer / rental). The registry has **no owner-count field** (24 fields, listed in Sprint 1), so using `Owner_Type`
would create training/serving skew. We still measure how much it would have added (D8 only):

In [9]:
d8 = mx.add_make_model(clean["d8_in"])
d8 = d8.dropna(subset=["owner_count"]).reset_index(drop=True)
rows = []
for label, num in [("without Owner_Type", ["age"]), ("with Owner_Type (owner_count)", ["age", "owner_count"])]:
    pipe = mx.build_pipeline(numeric=num, low_card=["fuel", "transmission"], make_enc="target", algo="xgb")
    cv = cross_validate(pipe, d8, d8.km, cv=KFold(5, shuffle=True, random_state=0), scoring=("neg_root_mean_squared_error", "r2"))
    rows.append({"D8-only 5-fold CV": label, "RMSE km": -cv["test_neg_root_mean_squared_error"].mean(), "R2": cv["test_r2"].mean()})
display(pd.DataFrame(rows))
print("Registry ownership (what we really get):", il.ownership.value_counts(normalize=True).round(3).to_dict())

,D8-only 5-fold CV,RMSE km,R2
0,without Owner_Type,"27,228.332",0.368
1,with Owner_Type (owner_count),"27,263.735",0.366


Registry ownership (what we really get): {'private': 0.829, 'leasing': 0.082, 'company': 0.048, 'dealer': 0.028, 'rental': 0.014}


## 3. Split protocol (fixed for every experiment below)

In [10]:
P = mx.add_make_model(pd.concat([clean["d7_uk"], clean["d8_in"], clean["d3_cardekho"]], ignore_index=True))
train_base, test = train_test_split(P, test_size=0.20, random_state=42, stratify=P["source"])
train_base, test = train_base.reset_index(drop=True), test.reset_index(drop=True)
d6 = mx.add_make_model(clean["d6_craigslist"])
print(f"pool 7+8+3: {len(P):,} -> train_base {len(train_base):,} | TEST {len(test):,} (same TEST for every experiment)")
print("TEST by source:", test.source.value_counts().to_dict())
print(f"Dataset 6 (cleaned, de-duplicated): {len(d6):,} rows — can only ever be added to TRAINING")

pool 7+8+3: 113,858 -> train_base 91,086 | TEST 22,772 (same TEST for every experiment)
TEST by source: {'d7_uk': 18716, 'd3_cardekho': 2617, 'd8_in': 1439}
Dataset 6 (cleaned, de-duplicated): 153,391 rows — can only ever be added to TRAINING


## 4. Experiment 1 — feature ablation (XGBoost, hold-out *validation* taken from `train_base`; TEST untouched)
Availability column: what the live system can provide for each feature.

In [11]:
fit_df, val_df = train_test_split(train_base, test_size=0.2, random_state=1)
FEATURE_SETS = [
    ("F0 age only",                          dict(numeric=["age"], low_card=[], make_enc=None, model_enc=None),                         "registry: shnat_yitzur"),
    ("F1 + fuel",                            dict(numeric=["age"], low_card=["fuel"], make_enc=None, model_enc=None),                    "registry: sug_delek_nm"),
    ("F2 + make",                            dict(numeric=["age"], low_card=["fuel"], make_enc="target", model_enc=None),                "registry: tozeret_nm"),
    ("F3 + model (make|model text)",         dict(numeric=["age"], low_card=["fuel"], make_enc="target", model_enc="target"),            "registry: kinuy_mishari — naming differs across datasets"),
    ("F4 F2 + transmission",                 dict(numeric=["age"], low_card=["fuel", "transmission"], make_enc="target", model_enc=None), "NOT in registry — spec lookup (API-Ninjas), coverage unknown"),
    ("F5 F4 + engine_cc",                    dict(numeric=["age", "engine_cc"], low_card=["fuel", "transmission"], make_enc="target"),   "NOT in registry resource — spec lookup"),
]
base_pred = np.full(len(val_df), fit_df.km.mean())
rows = [{"features": "B0 predict training mean", **mx.regression_metrics(val_df.km, base_pred), "runtime availability": "-"}]
for name, kw, avail in FEATURE_SETS:
    pipe = mx.build_pipeline(algo="xgb", **kw).fit(fit_df, fit_df.km)
    rows.append({"features": name, **mx.regression_metrics(val_df.km, pipe.predict(val_df)), "runtime availability": avail})
abl = pd.DataFrame(rows)
abl["ΔRMSE vs previous"] = abl.RMSE.diff()
display(abl)

,features,RMSE,MAE,R2,runtime availability,ΔRMSE vs previous
0,B0 predict training mean,"35,044.262","26,514.983",-0.000,-,NaN
1,F0 age only,"25,047.191","16,894.252",0.489,registry: shnat_yitzur,"-9,997.071"
2,F1 + fuel,"23,589.342","15,978.432",0.547,registry: sug_delek_nm,"-1,457.849"
3,F2 + make,"23,672.800","15,889.316",0.544,registry: tozeret_nm,83.458
4,F3 + model (make|model text),"22,881.404","15,350.837",0.574,registry: kinuy_mishari — naming differs acros...,-791.395
5,F4 F2 + transmission,"23,276.890","15,609.959",0.559,"NOT in registry — spec lookup (API-Ninjas), co...",395.486
6,F5 F4 + engine_cc,"22,613.800","15,226.309",0.584,NOT in registry resource — spec lookup,-663.090


## 5. Experiment 2 — categorical encoding (PDF requirement): OHE vs Target vs Frequency
Low-cardinality columns (fuel) always use One-Hot. For **make** (and optionally **make|model**, high cardinality) we compare One-Hot,
Target Encoding (sklearn `TargetEncoder`, cross-fitted inside each training fold) and Frequency Encoding. 3-fold CV on `train_base`.

In [12]:
cv3 = KFold(3, shuffle=True, random_state=0)
rows = []
for algo in ["xgb", "ridge"]:
    for make_enc in ["ohe", "target", "freq"]:
        for model_enc in [None, "target", "freq"]:
            pipe = mx.build_pipeline(numeric=["age"], low_card=["fuel"], make_enc=make_enc, model_enc=model_enc, algo=algo)
            r = cross_validate(pipe, train_base, train_base.km, cv=cv3, scoring=("neg_root_mean_squared_error", "neg_mean_absolute_error", "r2"))
            rows.append({"algo": algo, "make enc": make_enc, "make|model enc": model_enc or "-",
                         "RMSE": -r["test_neg_root_mean_squared_error"].mean(), "MAE": -r["test_neg_mean_absolute_error"].mean(), "R2": r["test_r2"].mean()})
enc = pd.DataFrame(rows).sort_values(["algo", "RMSE"])
display(enc)
best_make_enc = enc[(enc.algo == "xgb") & (enc["make|model enc"] == "-")].sort_values("RMSE").iloc[0]["make enc"]
print("Selected make encoding (XGB, no model column — see availability note):", best_make_enc)

,algo,make enc,make|model enc,RMSE,MAE,R2
10,ridge,ohe,target,"24,580.191","17,020.262",0.507
11,ridge,ohe,freq,"24,622.127","17,054.559",0.505
9,ridge,ohe,-,"24,674.610","17,091.092",0.503
16,ridge,freq,target,"24,818.405","17,174.456",0.497
13,ridge,target,target,"24,819.674","17,164.250",0.497
14,ridge,target,freq,"24,863.945","17,192.266",0.495
17,ridge,freq,freq,"24,870.188","17,201.844",0.495
15,ridge,freq,-,"24,913.094","17,230.617",0.493
12,ridge,target,-,"24,932.299","17,237.119",0.493
8,xgb,freq,freq,"22,388.811","15,146.431",0.591


Selected make encoding (XGB, no model column — see availability note): freq


## 6. Experiment 3 — algorithm comparison (PDF requirement)
Linear/Ridge (needs scaling) vs Random Forest vs XGBoost (no scaling), plus XGBoost on a log-target (mileage is right-skewed).
Config = `age + fuel + make` (the features guaranteed by the registry). 5-fold CV on `train_base`, then one fit on `train_base` evaluated on TEST.

In [13]:
cv5 = KFold(5, shuffle=True, random_state=0)
core = dict(numeric=["age"], low_card=["fuel"], make_enc=best_make_enc, model_enc=None)
rows, test_preds = [], {}
for algo in ["ridge", "rf", "xgb", "xgb_log"]:
    t0 = time.time()
    pipe = mx.build_pipeline(algo=algo, **core)
    r = cross_validate(pipe, train_base, train_base.km, cv=cv5, scoring=("neg_root_mean_squared_error", "neg_mean_absolute_error", "r2"))
    fitted = pipe.fit(train_base, train_base.km)
    test_preds[algo] = fitted.predict(test)
    m = mx.regression_metrics(test.km, test_preds[algo])
    rows.append({"algo": algo, "CV RMSE": -r["test_neg_root_mean_squared_error"].mean(), "CV MAE": -r["test_neg_mean_absolute_error"].mean(), "CV R2": r["test_r2"].mean(),
                 "TEST RMSE": m["RMSE"], "TEST MAE": m["MAE"], "TEST R2": m["R2"], "sec": time.time() - t0})
algs = pd.DataFrame(rows)
display(algs)
best_algo = algs.sort_values("CV RMSE").iloc[0]["algo"]
print("Selected algorithm by CV RMSE:", best_algo)

,algo,CV RMSE,CV MAE,CV R2,TEST RMSE,TEST MAE,TEST R2,sec
0,ridge,"24,911.773","17,229.114",0.493,"24,757.492","17,120.738",0.496,1.395
1,rf,"23,116.889","15,624.404",0.564,"22,778.502","15,406.491",0.574,16.229
2,xgb,"23,106.428","15,608.720",0.564,"22,743.859","15,381.456",0.575,24.085
3,xgb_log,"23,731.252","15,525.367",0.540,"23,359.272","15,276.982",0.552,22.909


Selected algorithm by CV RMSE: xgb


## 7. Experiment 4 — **7+8+3 vs 7+8+3+6**, same TEST set
* **A** = train on `train_base` (7+8+3).
* **B** = `train_base` + all cleaned Craigslist rows (equal weight).
* **C** = `train_base` + Craigslist at weight 0.25 (the US usage regime is different, so we down-weight it).
* **Reference** = Craigslist only (how far a purely US model transfers to the 7+8+3 test set).

Metrics: overall, per source, and **Israel-weighted** (test rows re-weighted by raking so make, fuel and age-bucket marginals match the Israeli registry sample;
rows whose make/fuel does not exist in Israel get weight 0). Paired bootstrap gives a 95% CI for ΔRMSE on identical rows.

In [14]:
d6_w = 0.25
algo = best_algo
def fit_model(train, weights=None):
    pipe = mx.build_pipeline(algo=algo, **core)
    kw = {} if weights is None else {"est__sample_weight": weights}
    return pipe.fit(train, train.km, **kw)

full_B = pd.concat([train_base, d6], ignore_index=True)
fitted = {
    "A: 7+8+3": fit_model(train_base),
    "B: 7+8+3 + D6 (w=1)": fit_model(full_B),
    "C: 7+8+3 + D6 (w=0.25)": fit_model(full_B, np.r_[np.ones(len(train_base)), np.full(len(d6), d6_w)]),
    "Ref: Craigslist only": fit_model(d6),
}
preds = {k: m.predict(test) for k, m in fitted.items()}

# Israel importance weights on TEST
age_bins = [-1, 2, 5, 8, 12, 17, 100]
tt = test.assign(age_b=pd.cut(test.age, age_bins, labels=False))
ilx = il.assign(age_b=pd.cut(il.age, age_bins, labels=False))
ilx = ilx[ilx.make.isin(set(test.make)) & ilx.fuel.isin(set(test.fuel))]
targets = {c: ilx[c].value_counts(normalize=True).to_dict() for c in ["make", "fuel", "age_b"]}
w_il = mx.raking_weights(tt, targets)
print(f"Israeli registry share whose make AND fuel exist in the TEST vocabulary: {(il.make.isin(set(test.make)) & il.fuel.isin(set(test.fuel))).mean():.1%}; rows with weight>0 in TEST: {(w_il>0).mean():.1%}")

rows = []
for name, p in preds.items():
    m = mx.regression_metrics(test.km, p)
    mi = mx.regression_metrics(test.km, p, weights=w_il)
    row = {"experiment": name, **{f"TEST {k}": v for k, v in m.items()}, **{f"Israel-weighted {k}": v for k, v in mi.items()}}
    for s in sorted(test.source.unique()):
        idx = (test.source == s).to_numpy()
        row[f"RMSE {s}"] = mx.regression_metrics(test.km[idx], p[idx])["RMSE"]
    rows.append(row)
exp4 = pd.DataFrame(rows).set_index("experiment")
display(exp4.T)

Israeli registry share whose make AND fuel exist in the TEST vocabulary: 75.5%; rows with weight>0 in TEST: 98.0%


experiment,A: 7+8+3,B: 7+8+3 + D6 (w=1),C: 7+8+3 + D6 (w=0.25),Ref: Craigslist only
TEST RMSE,"22,743.859","31,546.996","27,154.624","73,419.447"
TEST MAE,"15,381.456","20,588.126","17,765.659","61,831.187"
TEST R2,0.575,0.182,0.394,-3.430
Israel-weighted RMSE,"27,026.122","77,524.574","69,045.090","102,643.843"
Israel-weighted MAE,"18,998.521","60,699.175","52,124.734","91,419.094"
Israel-weighted R2,0.643,-1.939,-1.331,-4.152
RMSE d3_cardekho,"30,552.570","49,051.020","40,766.086","108,308.019"
RMSE d7_uk,"20,986.622","26,036.829","22,915.189","64,909.773"
RMSE d8_in,"27,565.888","50,560.413","42,624.679","95,760.598"


In [15]:
rows = []
for name in ["B: 7+8+3 + D6 (w=1)", "C: 7+8+3 + D6 (w=0.25)"]:
    d_all = mx.paired_bootstrap_delta(test.km, preds[name], preds["A: 7+8+3"])
    d_il = mx.paired_bootstrap_delta(test.km, preds[name], preds["A: 7+8+3"], weights=w_il)
    rows.append({"vs A": name, "ΔRMSE overall (km)": d_all["delta_rmse_mean"], "95% CI overall": f"[{d_all['ci95_low']:.0f}, {d_all['ci95_high']:.0f}]",
                 "ΔRMSE Israel-weighted (km)": d_il["delta_rmse_mean"], "95% CI Israel-w.": f"[{d_il['ci95_low']:.0f}, {d_il['ci95_high']:.0f}]"})
display(pd.DataFrame(rows))
print("negative Δ = adding Dataset 6 helps; a CI that contains 0 = no demonstrable difference")

# Mean signed error by age bucket (bias) for A vs B/C
tt["err_A"] = preds["A: 7+8+3"] - test.km
tt["err_B"] = preds["B: 7+8+3 + D6 (w=1)"] - test.km
tt["err_C"] = preds["C: 7+8+3 + D6 (w=0.25)"] - test.km
display(tt.groupby("age_b")[["err_A", "err_B", "err_C"]].mean().rename_axis("age bucket (0:0-2,1:3-5,2:6-8,3:9-12,4:13-17,5:18+)"))

,vs A,ΔRMSE overall (km),95% CI overall,ΔRMSE Israel-weighted (km),95% CI Israel-w.
0,B: 7+8+3 + D6 (w=1),"8,797.457","[8274, 9325]","51,046.481","[40670, 61212]"
1,C: 7+8+3 + D6 (w=0.25),"4,401.783","[3990, 4828]","42,373.165","[31909, 53155]"


negative Δ = adding Dataset 6 helps; a CI that contains 0 = no demonstrable difference


,err_A,err_B,err_C
"age bucket (0:0-2,1:3-5,2:6-8,3:9-12,4:13-17,5:18+)",,,
0,96.570,"1,742.129",607.265
1,347.219,"6,819.087","2,717.209"
2,180.186,"22,421.943","12,094.119"
3,-138.671,"46,652.121","30,875.504"
4,"2,320.691","59,437.241","48,763.658"
5,387.685,"73,191.331","68,303.491"


### 7.0 Israeli plausibility check (the TEST labels are UK/India, so they cannot tell us which regime is closer to Israel)
Every model is applied to the **real Israeli registry sample**. The model-implied **annual km** of a vehicle is `km(age) − km(age−1)` for the same make/fuel.
It is compared with public CBS figures. **These anchors are approximate and come from secondary sources that do not fully agree**
(CBS 2025 via [auto.co.il](https://www.auto.co.il/articles/car-news/local-news/car-mileage-israel-2025/): ≤3 yrs 16,700; 10–13 yrs 12,100; 20–29 yrs 9,100; fleet 14,700 km/yr;
CBS 2018 via [Jerusalem Post](https://www.pressreader.com/israel/jerusalem-post/20190917/281672551649195): fleet 16,100, ≤3 yrs 20,600;
2025 CBS items quoted elsewhere put *new privately-owned* cars at ~25,300 km/yr). Treat them as a **range**, not as labels. The primary CBS table should be checked before relying on a single number.

In [16]:
def implied_annual(m, d):
    cur = np.clip(m.predict(d), 0, None)
    prev = np.clip(m.predict(d.assign(age=(d.age - 1).clip(lower=0))), 0, None)
    return pd.Series(np.clip(cur - prev, 0, None), index=d.index)

il_feat = il.assign(model="unknown", make_model="unknown")
BANDS = {"age 1-3": (1, 3), "age 10-13": (10, 13), "age 20-25": (20, 25)}
CBS2025 = {"annual km age 1-3": 16700, "annual km age 10-13": 12100, "annual km age 20-25": 9100, "fleet mean (age>=1)": 14700}

def plausibility(m):
    inc = implied_annual(m, il_feat)
    r = {f"annual km {b}": inc[il.age.between(lo_, hi_)].mean() for b, (lo_, hi_) in BANDS.items()}
    r["fleet mean (age>=1)"] = inc[il.age >= 1].mean()
    r["EV (age>=1)"] = inc[(il.fuel == "electric") & (il.age >= 1)].mean()
    r["mean |dev| vs CBS-2025 anchors"] = float(np.mean([abs(r[k] / v - 1) for k, v in CBS2025.items()]))
    return r

plaus = pd.DataFrame({name: plausibility(m) for name, m in fitted.items()}).T
anch = pd.DataFrame([{**CBS2025, "EV (age>=1)": np.nan}], index=["CBS 2025 anchors (secondary source)"])
display(pd.concat([plaus, anch]))

,annual km age 1-3,annual km age 10-13,annual km age 20-25,fleet mean (age>=1),EV (age>=1),mean |dev| vs CBS-2025 anchors
A: 7+8+3,"7,653.274","8,627.459","3,646.419","7,601.847","4,803.147",0.478
B: 7+8+3 + D6 (w=1),"13,475.359","9,183.561","5,489.661","11,864.549","13,800.646",0.256
C: 7+8+3 + D6 (w=0.25),"12,141.857","9,949.570","4,967.443","11,291.248","11,524.278",0.284
Ref: Craigslist only,"22,414.402","8,313.978","5,263.596","13,329.175","20,217.209",0.292
CBS 2025 anchors (secondary source),"16,700.000","12,100.000","9,100.000","14,700.000",NaN,NaN


### 7.0b Trade-off curve: weight given to Dataset 6
For each weight: error on the 7+8+3 TEST (UK/India labels) vs. distance to the Israeli CBS anchors. If the two criteria disagree, that is the real finding.

In [17]:
rows = []
for w in [0.0, 0.05, 0.1, 0.25, 0.5, 1.0]:
    wts = np.r_[np.ones(len(train_base)), np.full(len(d6), w)]
    m = fit_model(full_B, wts) if w > 0 else fitted["A: 7+8+3"]
    p = m.predict(test)
    mt, mi = mx.regression_metrics(test.km, p), mx.regression_metrics(test.km, p, weights=w_il)
    pl = plausibility(m)
    rows.append({"D6 weight": w, "TEST RMSE": mt["RMSE"], "TEST MAE": mt["MAE"], "TEST R2": mt["R2"], "Israel-weighted RMSE": mi["RMSE"],
                 "fleet annual km": pl["fleet mean (age>=1)"], "annual km age 1-3": pl["annual km age 1-3"], "mean |dev| vs CBS-2025": pl["mean |dev| vs CBS-2025 anchors"]})
sweep = pd.DataFrame(rows)
display(sweep)
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(sweep["TEST RMSE"], sweep["mean |dev| vs CBS-2025"], "o-")
for _, r in sweep.iterrows():
    ax.annotate(f"w={r['D6 weight']}", (r["TEST RMSE"], r["mean |dev| vs CBS-2025"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("TEST RMSE on UK/India labels (km)"); ax.set_ylabel("mean |relative deviation| vs CBS-2025 anchors"); ax.grid(alpha=.3)
ax.set_title("Accuracy on UK/India vs plausibility for Israel"); plt.show()

,D6 weight,TEST RMSE,TEST MAE,TEST R2,Israel-weighted RMSE,fleet annual km,annual km age 1-3,mean |dev| vs CBS-2025
0,0.000,"22,743.859","15,381.456",0.575,"27,026.122","7,601.847","7,653.274",0.478
1,0.050,"24,353.513","16,207.089",0.513,"57,806.671","10,943.813","11,859.861",0.308
2,0.100,"25,298.823","16,701.112",0.474,"62,864.294","10,991.852","11,643.169",0.303
3,0.250,"27,154.624","17,765.659",0.394,"69,045.090","11,291.248","12,141.857",0.284
4,0.500,"29,130.216","18,973.399",0.303,"73,454.045","11,576.448","12,737.483",0.272
5,1.000,"31,546.996","20,588.126",0.182,"77,524.574","11,864.549","13,475.359",0.256


### 7.0c Is the plausibility gap an artefact of how `make` is encoded?
Frequency-encoding gives unseen makes (Kia, Mazda, Geely… ≈ half of the Israeli fleet) frequency 0, which could bias predictions. We repeat the check **without `make`** and with One-Hot make.

In [18]:
rows = []
variants = [("age+fuel (no make)", dict(numeric=["age"], low_card=["fuel"], make_enc=None)),
            ("age+fuel+make (OHE)", dict(numeric=["age"], low_card=["fuel"], make_enc="ohe")),
            ("age+fuel+make (freq)", dict(numeric=["age"], low_card=["fuel"], make_enc="freq"))]
for label, cfg in variants:
    for name, tr, w in [("A: 7+8+3", train_base, None), ("+D6 w=0.05", full_B, 0.05), ("+D6 w=0.25", full_B, 0.25), ("+D6 w=1", full_B, 1.0)]:
        wts = None if w is None else np.r_[np.ones(len(train_base)), np.full(len(d6), w)]
        m = mx.build_pipeline(algo=algo, **cfg).fit(tr, tr.km, **({} if wts is None else {"est__sample_weight": wts}))
        t, pl = mx.regression_metrics(test.km, m.predict(test)), plausibility(m)
        rows.append({"features": label, "training data": name, "TEST RMSE": t["RMSE"], "TEST R2": t["R2"], "annual km age 1-3": pl["annual km age 1-3"],
                     "annual km age 10-13": pl["annual km age 10-13"], "annual km age 20-25": pl["annual km age 20-25"],
                     "fleet mean": pl["fleet mean (age>=1)"], "mean |dev| vs CBS-2025": pl["mean |dev| vs CBS-2025 anchors"]})
variants_tbl = pd.DataFrame(rows)
display(variants_tbl)

,features,training data,TEST RMSE,TEST R2,annual km age 1-3,annual km age 10-13,annual km age 20-25,fleet mean,mean |dev| vs CBS-2025
0,age+fuel (no make),A: 7+8+3,"23,386.684",0.551,"10,266.652","6,859.324","1,249.996","7,637.945",0.540
1,age+fuel (no make),+D6 w=0.05,"26,970.059",0.402,"11,626.277","13,244.391","2,316.793","12,391.439",0.325
2,age+fuel (no make),+D6 w=0.25,"32,670.105",0.123,"12,829.953","11,903.203",720.801,"13,665.674",0.310
3,age+fuel (no make),+D6 w=1,"39,372.802",-0.274,"14,926.705","10,504.753",672.937,"14,302.869",0.298
4,age+fuel+make (OHE),A: 7+8+3,"22,770.743",0.574,"7,341.196","7,626.465","1,274.771","7,081.813",0.577
5,age+fuel+make (OHE),+D6 w=0.05,"24,352.299",0.513,"11,984.620","10,393.452","3,991.104","11,102.956",0.307
6,age+fuel+make (OHE),+D6 w=0.25,"27,151.309",0.394,"13,086.935","10,401.075","4,766.136","11,810.742",0.257
7,age+fuel+make (OHE),+D6 w=1,"31,516.703",0.184,"13,351.686","9,905.271","5,631.203","12,195.812",0.233
8,age+fuel+make (freq),A: 7+8+3,"22,743.859",0.575,"7,653.274","8,627.459","3,646.419","7,601.847",0.478
9,age+fuel+make (freq),+D6 w=0.05,"24,353.513",0.513,"11,859.861","10,641.358","3,952.642","10,943.813",0.308


Reading: dropping/changing `make` barely changes the picture, so the gap to the CBS anchors is **not** a make-encoding artefact. It comes from (i) the lower usage level of the UK/India listings and
(ii) thin support at old ages in the 7+8+3 pool (tree models cannot extrapolate: at ages 20–25 the 7+8+3-only model implies ~1–4k km/yr).

### 7.1 Is the Craigslist effect "more data" or "different market"?
Hold out 20% of Dataset 6 itself. If adding D6 helps *there* but not on the 7+8+3 test, the gain is market-specific, not general.

In [19]:
d6_tr, d6_te = train_test_split(d6, test_size=0.2, random_state=42)
algo_pipe = lambda: mx.build_pipeline(algo=algo, **core)
mA = algo_pipe().fit(train_base, train_base.km)
mB = algo_pipe().fit(pd.concat([train_base, d6_tr], ignore_index=True), pd.concat([train_base, d6_tr]).km)
rows = [
    {"model": "A (7+8+3)",  "RMSE on 7+8+3 TEST": mx.regression_metrics(test.km, mA.predict(test))["RMSE"], "RMSE on D6 hold-out": mx.regression_metrics(d6_te.km, mA.predict(d6_te))["RMSE"], "R2 on D6 hold-out": mx.regression_metrics(d6_te.km, mA.predict(d6_te))["R2"]},
    {"model": "B (+ D6 train)", "RMSE on 7+8+3 TEST": mx.regression_metrics(test.km, mB.predict(test))["RMSE"], "RMSE on D6 hold-out": mx.regression_metrics(d6_te.km, mB.predict(d6_te))["RMSE"], "R2 on D6 hold-out": mx.regression_metrics(d6_te.km, mB.predict(d6_te))["R2"]},
]
display(pd.DataFrame(rows))

,model,RMSE on 7+8+3 TEST,RMSE on D6 hold-out,R2 on D6 hold-out
0,A (7+8+3),"22,743.859","144,428.311",-1.441
1,B (+ D6 train),"30,769.038","71,900.202",0.395


## 8. Experiment 5 — how uncertain is a single mileage estimate? (quantile regression)
Model 2 will consume this estimate, so its uncertainty matters. 10%/90% quantile models trained on `train_base`; an honest 80% interval should cover ~80% of TEST rows.

In [20]:
from xgboost import XGBRegressor
qpreds = {}
for q in (0.1, 0.9):
    pipe = mx.build_pipeline(algo="xgb", **core)
    pipe.set_params(est=XGBRegressor(objective="reg:quantileerror", quantile_alpha=q, n_estimators=400, learning_rate=0.05, max_depth=5, subsample=0.8,
                                     colsample_bytree=0.8, min_child_weight=5, tree_method="hist", n_jobs=mx.N_JOBS, random_state=0))
    qpreds[q] = pipe.fit(train_base, train_base.km).predict(test)
lo, hi = np.minimum(qpreds[0.1], qpreds[0.9]), np.maximum(qpreds[0.1], qpreds[0.9])
inside = (test.km >= lo) & (test.km <= hi)
width = pd.Series(hi - lo, index=test.index)
res = pd.DataFrame({"coverage of 80% interval": inside.groupby(test.source).mean(), "median width (km)": width.groupby(test.source).median()})
res.loc["ALL"] = [inside.mean(), width.median()]
display(res)

,coverage of 80% interval,median width (km)
source,,
d3_cardekho,0.786,"64,207.898"
d7_uk,0.810,"42,056.715"
d8_in,0.789,"64,207.898"
ALL,0.806,"47,975.242"


## 9. Conclusions of the experiments (decision pending approval)
1. **Data quality.** After correcting units (miles→km), snapshot ages and duplicates, 7+8+3 gives 113,858 clean rows; Dataset 6 gives 153,391 (from 295,451 raw: ~1/3 were re-posted listings).
2. **Features.** Age carries almost all the signal (RMSE 35.0k → 25.0k km); fuel adds ~1.5k; make adds nothing measurable; transmission adds nothing (+0.4k worse); engine size ~0.7k but is **not** in the registry; `Owner_Type` adds nothing and does **not** map to `baalut`.
3. **Algorithm.** XGBoost ≈ Random Forest > log-target XGBoost > Ridge (TEST R² 0.575 / 0.574 / 0.552 / 0.496). Encoding choices matter ≤ ~1k km.
4. **Irreducible uncertainty.** Even the best model has TEST RMSE ≈ 22.7k km (R² ≈ 0.57); an honest 80% interval is ~48k km wide (coverage 80.6%). The estimate must be shown with its interval, and Model 2 must not treat it as exact.
5. **Dataset 6.** On the common TEST set (UK/India labels) adding Craigslist **hurts significantly** (ΔRMSE +4.4k at w=0.25, +8.8k at w=1; CIs exclude 0) and inflates older-car mileage (+22k to +73k km bias for ages ≥6).
   It helps only on its own market (D6 hold-out RMSE 144k → 72k) and improves coverage of Israeli makes (Kia, Nissan, Mazda, Mitsubishi) and of EVs.
6. **Israel.** No open Israeli odometer data was found, so no model can be validated on Israeli labels. Against *approximate* CBS anchors, the 7+8+3 model implies roughly **half** the Israeli annual mileage (≈7.6k vs ≈14.7–16.1k km/yr fleet);
   adding Dataset 6 (even at weight 0.05) moves it much closer (≈11–12k) at an accuracy cost on UK/India labels. The two criteria disagree; nothing in the data can settle it without Israeli ground truth.

## 9. Result tables used in the conclusion

In [21]:
print("Experiment 1 (ablation):"); display(abl)
print("Experiment 2 (encoding):"); display(enc)
print("Experiment 3 (algorithms):"); display(algs)
print("Experiment 4 (datasets):"); display(exp4.T)
print("Coverage of the Israeli fleet:"); display(cov)

Experiment 1 (ablation):


,features,RMSE,MAE,R2,runtime availability,ΔRMSE vs previous
0,B0 predict training mean,"35,044.262","26,514.983",-0.000,-,NaN
1,F0 age only,"25,047.191","16,894.252",0.489,registry: shnat_yitzur,"-9,997.071"
2,F1 + fuel,"23,589.342","15,978.432",0.547,registry: sug_delek_nm,"-1,457.849"
3,F2 + make,"23,672.800","15,889.316",0.544,registry: tozeret_nm,83.458
4,F3 + model (make|model text),"22,881.404","15,350.837",0.574,registry: kinuy_mishari — naming differs acros...,-791.395
5,F4 F2 + transmission,"23,276.890","15,609.959",0.559,"NOT in registry — spec lookup (API-Ninjas), co...",395.486
6,F5 F4 + engine_cc,"22,613.800","15,226.309",0.584,NOT in registry resource — spec lookup,-663.090


Experiment 2 (encoding):


,algo,make enc,make|model enc,RMSE,MAE,R2
10,ridge,ohe,target,"24,580.191","17,020.262",0.507
11,ridge,ohe,freq,"24,622.127","17,054.559",0.505
9,ridge,ohe,-,"24,674.610","17,091.092",0.503
16,ridge,freq,target,"24,818.405","17,174.456",0.497
13,ridge,target,target,"24,819.674","17,164.250",0.497
14,ridge,target,freq,"24,863.945","17,192.266",0.495
17,ridge,freq,freq,"24,870.188","17,201.844",0.495
15,ridge,freq,-,"24,913.094","17,230.617",0.493
12,ridge,target,-,"24,932.299","17,237.119",0.493
8,xgb,freq,freq,"22,388.811","15,146.431",0.591


Experiment 3 (algorithms):


,algo,CV RMSE,CV MAE,CV R2,TEST RMSE,TEST MAE,TEST R2,sec
0,ridge,"24,911.773","17,229.114",0.493,"24,757.492","17,120.738",0.496,1.395
1,rf,"23,116.889","15,624.404",0.564,"22,778.502","15,406.491",0.574,16.229
2,xgb,"23,106.428","15,608.720",0.564,"22,743.859","15,381.456",0.575,24.085
3,xgb_log,"23,731.252","15,525.367",0.540,"23,359.272","15,276.982",0.552,22.909


Experiment 4 (datasets):


experiment,A: 7+8+3,B: 7+8+3 + D6 (w=1),C: 7+8+3 + D6 (w=0.25),Ref: Craigslist only
TEST RMSE,"22,743.859","31,546.996","27,154.624","73,419.447"
TEST MAE,"15,381.456","20,588.126","17,765.659","61,831.187"
TEST R2,0.575,0.182,0.394,-3.430
Israel-weighted RMSE,"27,026.122","77,524.574","69,045.090","102,643.843"
Israel-weighted MAE,"18,998.521","60,699.175","52,124.734","91,419.094"
Israel-weighted R2,0.643,-1.939,-1.331,-4.152
RMSE d3_cardekho,"30,552.570","49,051.020","40,766.086","108,308.019"
RMSE d7_uk,"20,986.622","26,036.829","22,915.189","64,909.773"
RMSE d8_in,"27,565.888","50,560.413","42,624.679","95,760.598"


Coverage of the Israeli fleet:


,rows,make,fuel,age,make&fuel&age
D7 UK,"93,580.000",0.401,0.929,0.733,0.278
D8 India,"7,196.000",0.461,0.891,0.645,0.277
D3 Cardekho,"13,082.000",0.487,0.895,0.718,0.336
D7+D8+D3,"113,858.000",0.502,0.932,0.838,0.400
D6 Craigslist,"153,391.000",0.686,0.996,1.000,0.683
D7+D8+D3+D6,"267,249.000",0.829,1.000,1.000,0.829
